<a href="https://colab.research.google.com/github/Animesh-Maiti/Named-Entity-Recognition-for-Drug-and-Symptom-Extraction/blob/main/ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Install necessary libraries**

This cell installs all the required Python libraries for the NER project, including `torch`, `transformers`, `datasets`, `seqeval`, `scikit-learn`, `pandas`, and `tqdm`.

In [1]:

pip install torch transformers datasets seqeval scikit-learn pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e00a17a6d9334a06cc602311c9180026ca0c5df9ccf1b0662a6d893a5d597a0d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


### **Import Libraries**

This cell imports all the essential libraries and modules needed for building and training the Named Entity Recognition (NER) model. It includes `torch` for tensor operations, `AutoTokenizer`, `AutoModelForTokenClassification`, `TrainingArguments`, and `Trainer` from the `transformers` library for model handling, `Dataset`, `Features`, `Value`, `ClassLabel` from the `datasets` library for data management, `numpy` for numerical operations, and `classification_report` from `seqeval.metrics` for evaluating the model's performance.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset, Features, Value, ClassLabel
import numpy as np
from seqeval.metrics import classification_report

### **Define BIO Tags**

This cell defines the set of Beginning, Inside, and Outside (BIO) tags used for Named Entity Recognition. 'O' stands for 'Outside' (non-entity), 'B-DRUG' for 'Beginning of Drug entity', 'I-DRUG' for 'Inside of Drug entity', 'B-SYMPTOM' for 'Beginning of Symptom entity', and 'I-SYMPTOM' for 'Inside of Symptom entity'. These tags are crucial for labeling tokens in the dataset.

In [ ]:
bio_tags = [
 'O',
 'B-DRUG', 'I-DRUG',
 'B-SYMPTOM', 'I-SYMPTOM',
]


### **Read CoNLL formatted data**

This cell contains a function `read_conll_file` that reads data from a CoNLL-formatted file. CoNLL format is commonly used for sequence labeling tasks like NER, where each line typically contains a word and its corresponding label, with empty lines separating sentences. After defining the function, it uses it to load `train.tsv`, `devel.tsv`, and `test.tsv` files into separate lists of sentences and labels.

In [ ]:
# Assuming you have your data in CoNLL format
def read_conll_file(file_path):
    sentences, labels = [], []
    current_sentence, current_labels = [], []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                word, label = line.split('\t')
                current_sentence.append(word)
                current_labels.append(label)
            else:
                if current_sentence:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                    current_sentence, current_labels = [], []

    if current_sentence:
        sentences.append(current_sentence)
        labels.append(current_labels)

    return sentences, labels

train_sentences, train_labels = read_conll_file('train.tsv')
dev_sentences, dev_labels = read_conll_file('devel.tsv')
test_sentences, test_labels = read_conll_file('test.tsv')

### **Convert data to Hugging Face Dataset format**

This cell defines a function `convert_to_dataset` that transforms the raw sentences and labels into a Hugging Face `Dataset` object. It maps string labels (BIO tags) to numerical IDs and then constructs a `Dataset` with 'tokens' and 'ner_tags' features, specifying the `ClassLabel` type for `ner_tags` to ensure proper handling of categorical labels. Finally, it applies this conversion to the training, development, and test sets.

In [ ]:
def convert_to_dataset(sentences, labels, bio_tags):
    label_map = {tag: i for i, tag in enumerate(bio_tags)}

    # Convert string labels to IDs
    label_ids = [[label_map[label] for label in sentence_labels] for sentence_labels in labels]

    return Dataset.from_dict({
        "tokens": sentences,
        "ner_tags": label_ids
    }, features=Features({
        "tokens": [Value("string")],
        "ner_tags": [ClassLabel(names=bio_tags)]
    }))

train_dataset = convert_to_dataset(train_sentences, train_labels, bio_tags)
dev_dataset = convert_to_dataset(dev_sentences, dev_labels, bio_tags)
test_dataset = convert_to_dataset(test_sentences, test_labels, bio_tags)

### **Initialize Tokenizer**

This cell initializes the tokenizer for the pre-trained `dmis-lab/biobert-v1.1` model. The tokenizer is responsible for converting raw text into numerical input IDs that the BioBERT model can understand, handling tasks like tokenization, adding special tokens, and padding.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

### **Tokenize and Align Labels**

This cell defines the `tokenize_and_align_labels` function, which performs two critical steps for NER: tokenization and label alignment. It tokenizes the input sentences using the pre-trained BioBERT tokenizer, truncating and padding sequences to a fixed length. Crucially, it aligns the NER labels with the new subword tokens generated by the tokenizer, ensuring that each subword token has a corresponding label. It handles special tokens by assigning `-100` (ignored in loss calculation) and converts 'B-' tags to 'I-' for subsequent subword tokens of the same entity.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens have word_id set to None
            if word_idx is None:
                label_ids.append(-100)  # -100 will be ignored in loss calculation
            # For the first token of a word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For subsequent subword tokens
            else:
                # If it's a B- tag, convert to I- for continuation
                if bio_tags[label[word_idx]].startswith("B-"):
                    i_tag = "I-" + bio_tags[label[word_idx]][2:]
                    label_ids.append(bio_tags.index(i_tag))
                else:
                    label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

### **Apply Tokenization and Alignment to Datasets**

This cell applies the `tokenize_and_align_labels` function to the `train_dataset`, `dev_dataset`, and `test_dataset` using the `.map()` method. This processes each example in the datasets, generating tokenized inputs and properly aligned labels for model training and evaluation.

In [ ]:
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dev = dev_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True)

### **Load Pre-trained Model for Token Classification**

This cell loads a pre-trained BioBERT model (`dmis-lab/biobert-v1.1`) for token classification. It configures the model with the correct number of output labels, which corresponds to the total count of BIO tags defined earlier, enabling it to classify each token into one of the NER categories.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "dmis-lab/biobert-v1.1",
    num_labels=len(bio_tags)
)

### **Configure Training Arguments**

This cell sets up the `TrainingArguments` for the Hugging Face `Trainer`. It specifies various hyperparameters and training configurations, such as the output directory, evaluation and saving strategies per epoch, learning rate, batch sizes, number of training epochs, weight decay, and criteria for loading the best model (f1-score). It also includes `warmup_steps` and `logging_steps` for better training control.

In [ ]:
training_args = TrainingArguments(
    output_dir="./biobert-ner-custom",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    warmup_steps=100,        # ← replaces warmup_ratio
    logging_steps=50,        # ← kept this, still valid
)

### **Define Custom Metrics for Evaluation**

This cell defines the `compute_metrics` function, which calculates evaluation metrics during training and evaluation. It takes model predictions and true labels, converts them back from numerical IDs to BIO tags, removes ignored tokens (labeled as -100), and then generates a detailed classification report using `seqeval.metrics.classification_report`. It specifically extracts and returns precision, recall, and f1-score (micro average) for monitoring model performance.

In [ ]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [bio_tags[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [bio_tags[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = classification_report(true_labels, true_predictions, output_dict=True)

    # Extract metrics
    overall_f1 = results["micro avg"]["f1-score"]
    return {
        "precision": results["micro avg"]["precision"],
        "recall": results["micro avg"]["recall"],
        "f1": overall_f1,
    }

### **Initialize Data Collator**

This cell initializes a `DataCollatorForTokenClassification` from the `transformers` library. This collator is used to prepare batches of data for the model, performing tasks like dynamic padding to the longest sequence in each batch and handling the label IDs appropriately for token classification tasks.

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

### **Initialize and Start Training the Model**

This cell initializes the `Trainer` object, bringing together the model, training arguments, datasets, tokenizer, data collator, and the custom `compute_metrics` function. After setup, it calls `trainer.train()` to start the training process, which will fine-tune the BioBERT model on the provided training data and evaluate it on the development set.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()

### **Evaluate Model Performance on Test Set**

This cell evaluates the trained model on the `tokenized_test` dataset using `trainer.evaluate()`. It then prints the evaluation results, which typically include metrics like loss, precision, recall, and f1-score, providing an objective measure of the model's performance on unseen data.

In [ ]:
test_results = trainer.evaluate(tokenized_test)
print(f"Test results: {test_results}")

### **Generate Detailed Classification Report on Test Set**

This cell uses the trained `trainer` to make predictions on the `tokenized_test` dataset. It then processes these predictions and the true labels, aligning them and converting them back to BIO tags, while excluding ignored tokens. Finally, it prints a detailed `classification_report` from `seqeval`, providing per-entity precision, recall, and f1-scores, along with overall micro and macro averages.

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_test)
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [bio_tags[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [bio_tags[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

# Print detailed classification report
print(classification_report(true_labels, true_predictions))

### **Save Fine-tuned Model and Tokenizer**

This cell saves the fine-tuned BioBERT model and its corresponding tokenizer to a local directory named `./biobert-ner-final`. This allows for easy reloading and deployment of the trained model without needing to retrain it.

In [ ]:
model.save_pretrained("./biobert-ner-final")
tokenizer.save_pretrained("./biobert-ner-final")

### **Mount Google Drive**

This cell mounts your Google Drive to the Colab environment. This is necessary to save the trained model artifacts directly to your personal Drive storage, making them persistent even after the Colab runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### **Copy Model to Google Drive**

This cell uses the `shutil.copytree` function to copy the entire saved model directory (including the model weights and tokenizer files) from the local Colab environment to a specified location in your Google Drive (`/content/drive/MyDrive/biobert-ner-final`). This ensures that the trained model is permanently stored in your Drive.

In [ ]:
import shutil

# Copy the saved model folder to your Drive
shutil.copytree(
    "./biobert-ner-final",                          # source (local colab)
    "/content/drive/MyDrive/biobert-ner-final"      # destination (your drive)
)

print("Model saved to Google Drive! ✅")